# Notebook 08 — Uncertainty Estimation

Win projections are point estimates — but how confident should we be? This notebook adds bootstrapped confidence intervals to capture uncertainty from:

1. Stat noise: Small-sample players have unstable rate stats. We perturb each player's wOBA / FIP by a noise term scaled to their sample size.
2. Sample size penalty: Hitters with PA < 200 and pitchers with IP < 50 get amplified uncertainty via `sqrt(threshold / actual)`.
3. Low-confidence penalty

Method: For each of 1,000 bootstrap iterations:
- Perturb every player's key stat (wOBA for hitters, FIP for pitchers) by Gaussian noise scaled to their sample size
- Re-run the run estimator and Pythagorean formula
- Collect the distribution of projected wins

Output: Point estimate + 80% confidence interval (10th–90th percentile).

In [28]:
import os, sys, time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
DATA = os.path.join("..", "data")
np.random.seed(42)

master = pd.read_parquet(os.path.join(DATA, "master_players.parquet"))
park_factors = pd.read_parquet(os.path.join(DATA, "park_factors.parquet"))
hitters = master[master["player_type"] == "hitter"].copy()
pitchers = master[master["player_type"] == "pitcher"].copy()

print(f"Hitters: {len(hitters)}, Pitchers: {len(pitchers)}")

Hitters: 537, Pitchers: 656


## 1. Constants and noise model

wOBA noise: The year-to-year standard deviation of wOBA for qualified hitters is ~0.030. For a hitter with 600 PA, this is the baseline. For a hitter with 150 PA, we scale up by `sqrt(200/150) = 1.15×`.

FIP noise: Year-to-year FIP standard deviation for qualified starters is ~0.50. Same sample-size scaling for pitchers.

Low-confidence penalty: Added as extra variance. A n additional 0.5 wins of noise per low-confidence player in the roster.

In [30]:
# Run estimator constants (from NB06)
LG_WOBA = 0.315
WOBA_SCALE = 1.232
LG_R_PER_PA = 0.118
WOBA_REG_PA = 100
UNEARNED_RUN_FACTOR = 1.08
SIERA_WEIGHT, FIP_WEIGHT, ERA_WEIGHT = 0.40, 0.30, 0.30
LG_PITCHER_RATE = 4.10
TEAM_PA_162 = 6000
TEAM_IP_162 = 1458
STARTER_BENCH_SPLIT = 0.85
SP_IP_SHARE = 845 / TEAM_IP_162
RP_IP_SHARE = 1 - SP_IP_SHARE

# Noise parameters
WOBA_NOISE_BASE = 0.030     # baseline wOBA year-to-year sigma
RATE_NOISE_BASE = 0.50      # baseline pitching rate year-to-year sigma
PA_THRESHOLD = 200           # sample size threshold for hitters
IP_THRESHOLD = 50            # sample size threshold for pitchers
LOW_CONF_PENALTY_WINS = 0.5  # extra wins of uncertainty per low-conf player
N_BOOTSTRAP = 1000

print("Noise model parameters:")
print(f"  wOBA sigma (base): {WOBA_NOISE_BASE}")
print(f"  Rate sigma (base): {RATE_NOISE_BASE}")
print(f"  PA threshold:      {PA_THRESHOLD}")
print(f"  IP threshold:      {IP_THRESHOLD}")
print(f"  Low-conf penalty:  {LOW_CONF_PENALTY_WINS} wins/player")
print(f"  Bootstrap samples: {N_BOOTSTRAP}")

Noise model parameters:
  wOBA sigma (base): 0.03
  Rate sigma (base): 0.5
  PA threshold:      200
  IP threshold:      50
  Low-conf penalty:  0.5 wins/player
  Bootstrap samples: 1000


## 2. Bootstrap projection function

In [32]:
# Derive low_confidence flag from sample size
# Hitters with < 100 PA and pitchers with < 30 IP are flagged as low-confidence
hitters["low_confidence"] = hitters["PA"] < 100
pitchers["low_confidence"] = pitchers["IP"] < 30

n_lc_h = hitters["low_confidence"].sum()
n_lc_p = pitchers["low_confidence"].sum()
print(f"Low-confidence hitters: {n_lc_h}/{len(hitters)} ({n_lc_h/len(hitters):.0%})")
print(f"Low-confidence pitchers: {n_lc_p}/{len(pitchers)} ({n_lc_p/len(pitchers):.0%})")

# Build lookup tables
h_lookup = hitters.set_index("mlbam_id")[["wOBA", "xwOBA", "PA", "low_confidence"]].copy()
p_lookup = pitchers.set_index("mlbam_id")[["FIP", "xFIP", "ERA", "SIERA", "IP", "low_confidence"]].copy()

def _get_hitter_info(pid):
    """Return (wOBA, xwOBA, PA, low_confidence) for a hitter, with defaults."""
    if pid in h_lookup.index:
        row = h_lookup.loc[pid]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        woba = float(row["wOBA"]) if pd.notna(row["wOBA"]) else LG_WOBA
        xwoba = float(row["xwOBA"]) if pd.notna(row["xwOBA"]) else woba
        pa = float(row["PA"]) if pd.notna(row["PA"]) else 50
        lc = bool(row["low_confidence"]) if pd.notna(row["low_confidence"]) else True
        return woba, xwoba, pa, lc
    return LG_WOBA, LG_WOBA, 50, True

def _get_pitcher_info(pid):
    """Return (blended_rate, IP, low_confidence) for a pitcher, with defaults."""
    if pid in p_lookup.index:
        row = p_lookup.loc[pid]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        # SIERA/FIP/ERA blend (40/30/30) matching NB06
        siera = float(row["SIERA"]) if pd.notna(row.get("SIERA")) else None
        fip = None
        for col in ["FIP", "xFIP"]:
            if pd.notna(row.get(col)):
                fip = float(row[col])
                break
        era = float(row["ERA"]) if pd.notna(row.get("ERA")) else None
        tw, bl = 0, 0
        if siera is not None: bl += SIERA_WEIGHT * siera; tw += SIERA_WEIGHT
        if fip is not None: bl += FIP_WEIGHT * fip; tw += FIP_WEIGHT
        if era is not None: bl += ERA_WEIGHT * era; tw += ERA_WEIGHT
        rate = bl / tw if tw > 0 else LG_PITCHER_RATE
        ip = float(row["IP"]) if pd.notna(row["IP"]) else 10
        lc = bool(row["low_confidence"]) if pd.notna(row["low_confidence"]) else True
        return rate, ip, lc
    return LG_PITCHER_RATE, 10, True


def _regress(observed, sample_size, lg_mean, k):
    """Marcel-style regression toward league mean."""
    return (sample_size * observed + k * lg_mean) / (sample_size + k)


def _perturb_woba(woba, xwoba, pa, low_conf, rng):
    """Perturb regressed wOBA/xwOBA blend with scaled Gaussian noise."""
    blended = 0.5 * _regress(woba, pa, LG_WOBA, WOBA_REG_PA) + \
              0.5 * _regress(xwoba, pa, LG_WOBA, WOBA_REG_PA)
    scale = WOBA_NOISE_BASE
    if pa < PA_THRESHOLD:
        scale *= np.sqrt(PA_THRESHOLD / pa)
    if low_conf:
        scale += 0.008
    noisy = blended + rng.normal(0, scale)
    return np.clip(noisy, 0.150, 0.600)


def _perturb_rate(rate, ip, low_conf, rng):
    """Add scaled Gaussian noise to pitching rate."""
    scale = RATE_NOISE_BASE
    if ip < IP_THRESHOLD:
        scale *= np.sqrt(IP_THRESHOLD / ip)
    if low_conf:
        scale += 0.15
    noisy = rate + rng.normal(0, scale)
    return np.clip(noisy, 1.0, 8.0)


def bootstrap_projection(lineup_ids, bench_ids, sp_ids, rp_ids,
                          park_factor=100, n_iter=N_BOOTSTRAP, seed=42):
    """
    Run bootstrap projection using NB06 estimator logic.
    Returns: (point_estimate, win_samples array, ci_low_80, ci_high_80)
    """
    rng = np.random.default_rng(seed)
    
    # Pre-fetch player info
    lineup_info = [_get_hitter_info(pid) for pid in lineup_ids]
    bench_info = [_get_hitter_info(pid) for pid in bench_ids]
    sp_info = [_get_pitcher_info(pid) for pid in sp_ids]
    rp_info = [_get_pitcher_info(pid) for pid in rp_ids]
    
    n_low_conf = sum(1 for *_, lc in lineup_info + bench_info if lc) + \
                 sum(1 for _, _, lc in sp_info + rp_info if lc)
    team_lc_noise = n_low_conf * LOW_CONF_PENALTY_WINS
    
    pf_adj = (park_factor / 100 + 1) / 2
    starter_pa_total = TEAM_PA_162 * STARTER_BENCH_SPLIT
    bench_pa_total = TEAM_PA_162 * (1 - STARTER_BENCH_SPLIT)
    
    # PA/IP weights from actual playing time
    lineup_pa = [pa for _, _, pa, _ in lineup_info]
    tot_lineup_pa = sum(lineup_pa) or 1
    bench_pa = [pa for _, _, pa, _ in bench_info]
    tot_bench_pa = sum(bench_pa) or 1
    sp_ip = [ip for _, ip, _ in sp_info]
    tot_sp_ip = sum(sp_ip) or 1
    rp_ip = [ip for _, ip, _ in rp_info]
    tot_rp_ip = sum(rp_ip) or 1
    
    sp_ip_total = TEAM_IP_162 * SP_IP_SHARE
    rp_ip_total = TEAM_IP_162 * RP_IP_SHARE
    
    # Point estimate (no noise, using regressed blend)
    rs_point = 0.0
    for i, (woba, xwoba, pa, _) in enumerate(lineup_info):
        blended = 0.5 * _regress(woba, pa, LG_WOBA, WOBA_REG_PA) + \
                  0.5 * _regress(xwoba, pa, LG_WOBA, WOBA_REG_PA)
        alloc = starter_pa_total * (lineup_pa[i] / tot_lineup_pa)
        rs_point += ((blended - LG_WOBA) / WOBA_SCALE + LG_R_PER_PA) * alloc
    if bench_info:
        for i, (woba, xwoba, pa, _) in enumerate(bench_info):
            blended = 0.5 * _regress(woba, pa, LG_WOBA, WOBA_REG_PA) + \
                      0.5 * _regress(xwoba, pa, LG_WOBA, WOBA_REG_PA)
            alloc = bench_pa_total * (bench_pa[i] / tot_bench_pa)
            rs_point += ((blended - LG_WOBA) / WOBA_SCALE + LG_R_PER_PA) * alloc
    else:
        rs_point += LG_R_PER_PA * bench_pa_total
    rs_point *= pf_adj
    
    ra_point = 0.0
    for i, (rate, ip, _) in enumerate(sp_info):
        ra_point += (rate / 9) * sp_ip_total * (sp_ip[i] / tot_sp_ip)
    for i, (rate, ip, _) in enumerate(rp_info):
        ra_point += (rate / 9) * rp_ip_total * (rp_ip[i] / tot_rp_ip)
    ra_point *= UNEARNED_RUN_FACTOR * pf_adj
    
    # Pythagenpat point estimate
    pyth_exp = ((rs_point + ra_point) / 162) ** 0.287
    rs_k = rs_point ** pyth_exp
    point_wins = round((rs_k / (rs_k + ra_point ** pyth_exp)) * 162)
    
    # Bootstrap iterations
    win_samples = np.empty(n_iter)
    
    for b in range(n_iter):
        rs = 0.0
        for i, (woba, xwoba, pa, lc) in enumerate(lineup_info):
            w_noisy = _perturb_woba(woba, xwoba, pa, lc, rng)
            alloc = starter_pa_total * (lineup_pa[i] / tot_lineup_pa)
            rs += ((w_noisy - LG_WOBA) / WOBA_SCALE + LG_R_PER_PA) * alloc
        if bench_info:
            for i, (woba, xwoba, pa, lc) in enumerate(bench_info):
                w_noisy = _perturb_woba(woba, xwoba, pa, lc, rng)
                alloc = bench_pa_total * (bench_pa[i] / tot_bench_pa)
                rs += ((w_noisy - LG_WOBA) / WOBA_SCALE + LG_R_PER_PA) * alloc
        else:
            rs += LG_R_PER_PA * bench_pa_total
        rs *= pf_adj
        
        ra = 0.0
        for i, (rate, ip, lc) in enumerate(sp_info):
            r_noisy = _perturb_rate(rate, ip, lc, rng)
            ra += (r_noisy / 9) * sp_ip_total * (sp_ip[i] / tot_sp_ip)
        for i, (rate, ip, lc) in enumerate(rp_info):
            r_noisy = _perturb_rate(rate, ip, lc, rng)
            ra += (r_noisy / 9) * rp_ip_total * (rp_ip[i] / tot_rp_ip)
        ra *= UNEARNED_RUN_FACTOR * pf_adj
        
        # Team-level low-confidence noise
        team_noise = rng.normal(0, team_lc_noise) if team_lc_noise > 0 else 0
        
        # Pythagenpat
        pyth_exp = ((rs + ra) / 162) ** 0.287
        rs_k = rs ** pyth_exp
        win_pct = rs_k / (rs_k + ra ** pyth_exp)
        win_samples[b] = win_pct * 162 + team_noise
    
    ci_low = np.percentile(win_samples, 10)
    ci_high = np.percentile(win_samples, 90)
    
    return point_wins, win_samples, ci_low, ci_high


print("Bootstrap projection function defined.")

Low-confidence hitters: 76/537 (14%)
Low-confidence pitchers: 182/656 (28%)
Bootstrap projection function defined.


## 3. Assemble 2025 Dodgers roster

We'll use the 2025 Dodgers as our test case. Roster splits:
- **Lineup (9):** Top 9 hitters by PA (everyday starters)
- **Bench (6):** Remaining hitters
- **Starting pitchers (5):** Top 5 pitchers by GS
- **Relief pitchers:** All remaining pitchers

Dodgers park factor (5yr): **99** (slightly pitcher-friendly).

In [34]:
# Dodgers roster from master_players
dodgers = master[master["team"] == "Los Angeles Dodgers"].copy()
dodgers_h = dodgers[dodgers["player_type"] == "hitter"].sort_values("PA", ascending=False)
dodgers_p = dodgers[dodgers["player_type"] == "pitcher"].sort_values("IP", ascending=False)

# Lineup: top 9 by PA
lineup_ids = dodgers_h["mlbam_id"].head(9).tolist()
bench_ids = dodgers_h["mlbam_id"].iloc[9:].tolist()

# SP: top 5 by GS (starting games)
dodgers_sp = dodgers_p.sort_values("GS", ascending=False)
sp_ids = dodgers_sp["mlbam_id"].head(5).tolist()
rp_ids = dodgers_sp["mlbam_id"].iloc[5:].tolist()

# Park factor
dodgers_pf = park_factors.loc[park_factors["team"] == "Dodgers", "park_factor_5yr"].values[0]

print("=== 2025 Dodgers Test Roster ===\n")
print("LINEUP (9):")
for i, pid in enumerate(lineup_ids, 1):
    row = dodgers_h[dodgers_h["mlbam_id"] == pid].iloc[0]
    print(f"  {i}. {row['name']:<22} PA={int(row['PA']):>3}  wOBA={row['wOBA']:.3f}")

print(f"\nBENCH ({len(bench_ids)}):")
for pid in bench_ids:
    row = dodgers_h[dodgers_h["mlbam_id"] == pid].iloc[0]
    print(f"     {row['name']:<22} PA={int(row['PA']):>3}  wOBA={row['wOBA']:.3f}")

print(f"\nSTARTING PITCHERS ({len(sp_ids)}):")
for pid in sp_ids:
    row = dodgers_p[dodgers_p["mlbam_id"] == pid].iloc[0]
    print(f"     {row['name']:<22} IP={row['IP']:>5.1f}  FIP={row['FIP']:.2f}")

print(f"\nRELIEF PITCHERS ({len(rp_ids)}):")
for pid in rp_ids:
    row = dodgers_p[dodgers_p["mlbam_id"] == pid].iloc[0]
    print(f"     {row['name']:<22} IP={row['IP']:>5.1f}  FIP={row['FIP']:.2f}")

print(f"\nPark factor (5yr): {dodgers_pf}")

=== 2025 Dodgers Test Roster ===

LINEUP (9):
  1. Shohei Ohtani          PA=727  wOBA=0.418
  2. Mookie Betts           PA=663  wOBA=0.318
  3. Freddie Freeman        PA=627  wOBA=0.370
  4. Andy Pages             PA=624  wOBA=0.332
  5. Teoscar Hernández      PA=546  wOBA=0.315
  6. Michael Conforto       PA=486  wOBA=0.287
  7. Will Smith             PA=436  wOBA=0.389
  8. Max Muncy              PA=388  wOBA=0.366
  9. Tommy Edman            PA=377  wOBA=0.284

BENCH (5):
     Miguel Rojas           PA=317  wOBA=0.312
     Enrique Hernández      PA=256  wOBA=0.268
     Hyeseong Kim           PA=170  wOBA=0.306
     Dalton Rushing         PA=155  wOBA=0.256
     Alex Freeland          PA= 97  wOBA=0.273

STARTING PITCHERS (5):
     Yoshinobu Yamamoto     IP=173.2  FIP=2.94
     Dustin May             IP=132.1  FIP=4.88
     Clayton Kershaw        IP=112.2  FIP=3.55
     Tyler Glasnow          IP= 90.1  FIP=3.76
     Emmet Sheehan          IP= 73.1  FIP=2.93

RELIEF PITCHERS (20):
  